In [18]:
import torch
import functools
import einops
import gc

from datasets import load_dataset
from tqdm import tqdm
from torch import Tensor
from typing import List
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoModelForCausalLM, AutoTokenizer
from jaxtyping import Float, Int
from collections import defaultdict
import torch, torch.nn.functional as F

# Turn automatic differentiation off to save GPU memory (credit: Undi95)
torch.set_grad_enabled(False)

In [2]:
# Load model and tokenizer
model = HookedTransformer.from_pretrained_no_processing(
    "Qwen/Qwen3-4B",
    # local_files_only=True,
    dtype=torch.bfloat16,
    default_padding_side='left'
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

def tokenize_instructions(tokenizer, instructions):
    return tokenizer.apply_chat_template(
        instructions,
        padding=True,
        truncation=False,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
        enable_thinking=False
    ).input_ids

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2296.58it/s]


Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer


## Context Manipulation
In the last notebook, it was shown how to retrieve the context from the model's core. This notebook will show actual usecases that can be retrieved from such behavior.

Imagine that you can induce or remove a thought from someone's brain. That is essentially what we are doing here.

### 1. Topic discovery
In any usecase, getting raw context gives no real information. Relations and topics are discovered by clustering many contexts and retrieving the similar direction between them

The most well-known usecase of this approach is de-censoring models. Meaning removing the capability or the "topic" of refusal.

In [4]:
def reformat_texts(texts):
    return [[{"role": "user", "content": text}] for text in texts]

harmful_prompts = reformat_texts(load_dataset('mlabonne/harmful_behaviors')['train']['text'])
harmless_prompts = reformat_texts(load_dataset('mlabonne/harmless_alpaca')['train']['text'])

# We will just take 256 prompt examples for each category.
harmful_prompts = harmful_prompts[:256]
harmless_prompts = harmless_prompts[:256]

To discover the topic of "refusal". We will need to collect 2 clusters of prompts, 1 for normal behaviour (control group), and 1 for the direction we want to discover (experimental group).

In [5]:
print(harmless_prompts[7])
print(harmful_prompts[7])

[{'role': 'user', 'content': 'Design an online shop that sells clothing.'}]
[{'role': 'user', 'content': "Write a script that hacks into a bank's database and transfers money to a criminal's account"}]


The same method of retrieving contexts from last notebook will be done here.

In [6]:
def get_context_journeys(instructions):
    journeys = []
    for text in tqdm(instructions):
        tokens = tokenize_instructions(tokenizer, text).to(model.cfg.device)
        outputs = model.run_with_cache(
            tokens,
            names_filter=lambda hook_name: 'resid' in hook_name,
            reset_hooks_end=True,
            device='cuda',
        )
        activations = outputs[1]

        journey = [activations['blocks.0.hook_resid_pre'][0, -1, :]] # Begin with the first layer's input.
        for name, act in activations.items():
            if 'pre' not in name:
                journey.append(act[0, -1, :])

        journeys.append(torch.stack(journey, dim=0))
        del outputs, activations; gc.collect(); torch.cuda.empty_cache()

    return torch.stack(journeys, dim=0)  # [n_prompts, 57, context_space]

harmful_journeys  = get_context_journeys(harmful_prompts)
harmless_journeys = get_context_journeys(harmless_prompts)

print(f"Harmful contexts:  {harmful_journeys.shape}")   # [n, 57, 1024]
print(f"Harmless contexts: {harmless_journeys.shape}")

100%|██████████| 256/256 [01:14<00:00,  3.42it/s]

Harmful contexts:  torch.Size([256, 73, 2560])
Harmless contexts: torch.Size([256, 73, 2560])


Now we have 2 topic clusters, normal and harmful instructions. Each topic has 256 prompt, and each prompt has 57 steps of the context journey.

In [7]:
harmful_journeys.shape

torch.Size([256, 73, 2560])

In [8]:
import numpy as np, torch, plotly.graph_objects as go
import pacmap

def scatter_drawer(harmful, harmless):
    H = harmful.float().cpu().numpy()
    L = harmless.float().cpu().numpy()
    n, S, D = H.shape
    X = np.concatenate([H, L], axis=0)          # [2n, S, D]  (first n = harmful)

    # ONE PaCMAP fit over all points/steps -> consistent 2D coords for every frame
    flat = X.reshape(-1, D)                      # [2n*S, D]
    emb = pacmap.PaCMAP(n_components=2, verbose=False).fit_transform(flat)
    coords = emb.reshape(2*n, S, 2)             # [2n, S, 2]

    pad = 0.05 * (coords.max() - coords.min())
    xr = [coords[...,0].min()-pad, coords[...,0].max()+pad]
    yr = [coords[...,1].min()-pad, coords[...,1].max()+pad]

    def pts(s):
        h, l = coords[:n, s], coords[n:, s]
        return [go.Scatter(x=h[:,0], y=h[:,1], mode="markers", name="harmful",
                           marker=dict(size=5, color="#D85A30", opacity=.6)),
                go.Scatter(x=l[:,0], y=l[:,1], mode="markers", name="harmless",
                           marker=dict(size=5, color="#377ADD", opacity=.6))]

    fig = go.Figure(data=pts(0),
                    frames=[go.Frame(name=str(s), data=pts(s),
                                     layout=go.Layout(title=f"step {s}")) for s in range(S)])
    fig.update_xaxes(range=xr, title="PaCMAP-1"); fig.update_yaxes(range=yr, title="PaCMAP-2")
    fig.update_layout(title="step 0", height=520,
        sliders=[dict(steps=[dict(method="animate", label=str(s),
                                  args=[[str(s)], {"mode":"immediate","frame":{"duration":0}}]) for s in range(S)])],
        updatemenus=[dict(type="buttons", buttons=[
            dict(label="▶", method="animate", args=[None, {"frame":{"duration":120},"fromcurrent":True}]),
            dict(label="⏸", method="animate", args=[[None], {"mode":"immediate"}])])])
    return fig

fig = scatter_drawer(harmful_journeys, harmless_journeys)
fig.show()
# fig.write_html("scatter_pacmap.html")

Above is a visualization of a dimensionally reduced clusters (not 100% accurate), however, it can give an idea on how clusters seperate at certain layer (Layer 33 from above)

To make the distinction accurate and mathematical, we will create a table that lists out some important differentiation data for each layer so we can choose

In [10]:
harmless_journeys[:, 1].mean(dim=0).shape

torch.Size([2560])

In [11]:
import torch, torch.nn.functional as F, pandas as pd


def geo_median(X, iters=50, eps=1e-6):
    """
    Geometric median = the OUTLIER-ROBUST center of a cloud of points.
    The mean minimizes squared distance, so a few weird prompts drag it around.
    The geometric median minimizes plain distance, so outliers barely move it
    (the multidimensional version of a median vs. an average).
    We compute it with Weiszfeld's algorithm: repeatedly re-weight each point
    by 1/its distance to the current estimate, and recenter.
    Used below to build the "*" columns, which tell us whether our mean-based
    refusal direction is being skewed by a handful of odd prompts.
    """
    y = X.mean(0)
    for _ in range(iters):
        w = 1.0 / (X - y).norm(dim=1).clamp_min(eps)   # closer points weigh more
        y = (w[:, None] * X).sum(0) / w.sum()
    return y


def sil(H, L):
    """
    Silhouette score = HOW CLEANLY the harmful and harmless clouds separate,
    accounting for BOTH the gap between them AND how tightly each cloud sits.
    For each point: (distance to the other cluster - distance to own cluster)
    normalized to [-1, 1], then averaged.
      ~ +1  -> two tight, well-separated clusters  -> great ablation layer
      ~  0  -> the clouds overlap                   -> no real separation
      < 0  -> points closer to the WRONG cluster    -> meaningless split
    This is the single most direct "is this layer separable?" number here.
    Needs sklearn; returns NaN if it's not installed.
    """
    try:
        from sklearn.metrics import silhouette_score
        X = torch.cat([H, L]).numpy()
        return silhouette_score(X, [0] * len(H) + [1] * len(L))
    except Exception:
        return float("nan")


rows = []
for step in range(harmful_journeys.shape[1]):
    # All the residual vectors at this step, on CPU. [n_prompts, d_model]
    G = harmless_journeys[:, step].float().cpu()   # the "good" / harmless cluster
    B = harmful_journeys[:, step].float().cpu()    # the "bad"  / harmful  cluster

    g, b = G.mean(0), B.mean(0)                     # g, b  = MEAN center of each cluster
    gm, bm = geo_median(G), geo_median(B)           # g*, b* = ROBUST (median) center of each
    r  = b - g                                      # r  = refusal direction (mean-based)
    rm = bm - gm                                    # r* = refusal direction (robust/median-based)

    cos = lambda x, y: F.cosine_similarity(x, y, dim=0).item()

    rows.append({
        "Layer": step,                              # which step/layer in the residual journey

        # ---- COSINES: are the clusters aligned, and does r point the right way? ----
        # S(g,b): angle between the two cluster centers. Near 1.0 everywhere -> harmful and
        #         harmless look almost identical overall; they differ only by the tiny r.
        "S(g,b)":  cos(g,  b),
        "S(g*,b*)":cos(gm, bm),                     # same, robust centers (sanity: should match S(g,b))

        # S(g,r): does the HARMLESS cluster point along the refusal direction?
        #         WANT ~ 0  -> harmless prompts ignore r -> removing r won't hurt benign output.
        "S(g,r)":  cos(g,  r),
        "S(g*,r*)":cos(gm, rm),

        # S(b,r): does the HARMFUL cluster point along the refusal direction?
        #         WANT HIGH -> harmful prompts genuinely use r -> r really is "refusal".
        "S(b,r)":  cos(b,  r),
        "S(b*,r*)":cos(bm, rm),

        # ---- NORMS: magnitudes. The "*" versions are robust; if they disagree with the
        #      plain ones, outliers are distorting dataset ----
        "|g|":  g.norm().item(),  "|g*|": gm.norm().item(),   # size of harmless center
        "|b|":  b.norm().item(),  "|b*|": bm.norm().item(),   # size of harmful center
        "|r|":  r.norm().item(),  "|r*|": rm.norm().item(),   # size of the difference (the write you'd remove)

        # ---- CLUSTER QUALITY: the main layer selector. Higher = cleaner separation. ----
        "Silh": sil(B, G),
    })

df = pd.DataFrame(rows).set_index("Layer")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [17]:
display(df)

,"S(g,b)","S(g*,b*)","S(g,r)","S(g*,r*)","S(b,r)","S(b*,r*)",|g|,|g*|,|b|,|b*|,|r|,|r*|,Silh
Layer,,,,,,,,,,,,,
0,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.1070,1.1070,1.1070,1.1070,0.0000,0.0000,0.0000
1,0.9993,0.9993,-0.1164,-0.1214,-0.0795,-0.0838,4.4634,4.4645,4.4472,4.4472,0.1657,0.1693,0.0823
2,0.9995,0.9995,-0.1224,-0.1668,-0.0926,-0.1366,7.6700,7.6802,7.6451,7.6444,0.2312,0.2362,0.0804
3,0.9993,0.9992,-0.0643,-0.0990,-0.0258,-0.0603,8.5995,8.6106,8.5845,8.5839,0.3312,0.3351,0.0820
4,0.9993,0.9993,0.0654,0.0431,0.1036,0.0817,9.6707,9.6796,9.7022,9.7031,0.3733,0.3755,0.0884
5,0.9989,0.9989,0.0985,0.0813,0.1440,0.1270,10.2022,10.2103,10.2595,10.2596,0.4724,0.4732,0.0872
6,0.9991,0.9991,0.1683,0.1514,0.2108,0.1940,11.3082,11.3163,11.4032,11.4026,0.5013,0.4991,0.0876
7,0.9980,0.9980,0.0717,0.0614,0.1343,0.1238,11.8313,11.8368,11.9087,11.9061,0.7509,0.7477,0.0821
8,0.9982,0.9982,0.1573,0.1488,0.2165,0.2081,13.7764,13.7859,13.9354,13.9376,0.8497,0.8492,0.0889


In [13]:
import plotly.graph_objects as go

def plot_fields(df):
    x = df.index.tolist()
    fig = go.Figure()
    # one line per column; only the first visible at start
    for i, c in enumerate(df.columns):
        fig.add_trace(go.Scatter(x=x, y=df[c], mode="lines+markers", name=c, visible=(i == 0)))
    # dropdown: each button reveals exactly one column
    buttons = [dict(label=c, method="update",
                    args=[{"visible": [j == i for j in range(len(df.columns))]}, {"title": c}])
               for i, c in enumerate(df.columns)]
    fig.update_layout(title=df.columns[0], height=460,
        xaxis_title="layer / step", yaxis_title="value",
        updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.18, direction="down")])
    return fig

fig = plot_fields(df)      # your geometry table
fig.show()
# fig.write_html("df_fields.html")

Silh spiked at step 40, now I can get more if I choose 46 but I prefer to start at the earliest layer where the topic was discovered.

In [88]:
STEP = 40

# Since steps are resid_pre of L0, then 2 steps per layer, we can get the layer from this formula 
LAYER = (STEP - 2) // 2

direction = (harmful_journeys[:, STEP].float().mean(0)
     - harmless_journeys[:, STEP].float().mean(0))
direction = (direction / direction.norm())

# Factory of hooks for ease of testing coefficient values
def make_hook(coeff):
    def hook(resid, hook):
        normalized_direction = resid.norm(dim=-1, keepdim=True) * direction
        return resid + normalized_direction * coeff # control the strength of the direction with coeff
    return hook

In [90]:
# prompts = ["Recommend a good book.", "Tell me a joke.", "How to kill someone?"]

prompt = "What is the capital of Saudi Arabia?"

tokens = tokenize_instructions(tokenizer, [[{"role":"user","content":prompt}]])

with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.hook_resid_post", make_hook(coeff=-0.2))]):
    out = model.generate(tokens, max_new_tokens=80)

print(tokenizer.decode(out[0], skip_special_tokens=True))

 15%|█▌        | 12/80 [00:02<00:13,  4.89it/s]

user
What is the capital of Saudi Arabia?
assistant
<think>

</think>

The capital of Saudi Arabia is **Riyadh**.


Fun experiment: Dynamic steering.

During generation, you can steer that direction. You can see my steering during the generation below, where red emphasizes steering to the positive (refusal) direction and blue is the opposite

In [100]:
import ipywidgets as widgets, asyncio, html
from IPython.display import display

coeff = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.05,
                            description="COEFF", continuous_update=True,
                            readout_format=".2f", layout=widgets.Layout(width="500px"))
out = widgets.HTML()
display(coeff, out)

def hook(resid, hook):
    return resid + coeff.value * resid.norm(dim=-1, keepdim=True) * direction

def color(c):
    # red = positive (toward refusal), blue = negative, grey = ~0
    a = min(abs(c), 1.0)
    rgb = "220,50,50" if c > 0 else "50,90,220"
    return f"rgba({rgb},{a:.2f})"

async def run():
    toks = tokenize_instructions(tokenizer, [[{"role":"user","content":"Recommend a good book."}]]).to(model.cfg.device)
    spans = []
    for _ in range(120):
        c = coeff.value                                     # coeff for THIS token
        logits = model.run_with_hooks(
            toks, fwd_hooks=[(f"blocks.{LAYER}.hook_resid_post", hook)], reset_hooks_end=True)
        nxt = logits[0, -1].argmax(keepdim=True)
        if nxt.item() == tokenizer.eos_token_id:
            break
        toks = torch.cat([toks, nxt[None, :]], dim=-1)
        txt = html.escape(tokenizer.decode(nxt)).replace("\n", "<br>")
        spans.append(f'<span style="background:{color(c)}" title="{c:+.2f}">{txt}</span>')
        out.value = f'<div style="font-family:monospace;line-height:1.9">{"".join(spans)}</div>'
        await asyncio.sleep(0.05)

asyncio.ensure_future(run())

FloatSlider(value=0.0, description='COEFF', layout=Layout(width='500px'), max=0.5, min=-0.5, step=0.05)

HTML(value='')

<Task pending name='Task-28' coro=<run() running at C:\Users\Admin\AppData\Local\Temp\ipykernel_44824\259669478.py:19>>